# Study report: stack-guard

The study measures how much the MPU stack guard adds to the IRQ entry,
the ISR to thread wake and the thread to thread latency. The variant
`no-mpu-guard` removes the stack guard. The study imports the three
experiments of the parent study kconfig-factors.

This notebook reads one built instance of the study (`study.py analysis
build`). `study.py analysis notebooks` runs it and writes the HTML into the
instance folder. In the figures: hover for values, drag to zoom, click a
legend entry to hide a series, and use the menu above a figure to select
the metric. Commit this file without outputs.

In [ ]:
# Parameters. The instance folder: LATBENCH_INSTANCE ("study.py analysis notebooks" sets it), else the
# folder of this notebook when it is the executed copy in <instance>/notebooks/, else the instance of this
# study in LATBENCH_RESULTS (a results root). The safety repo: LATBENCH_REPO, else the design repo of the
# lab space of the instance (.lab/config), else the repo of the current folder.
import configparser
import os
import subprocess
import sys
from pathlib import Path

here = Path.cwd()
INSTANCE = Path(os.environ.get("LATBENCH_INSTANCE") or next(
    (d for d in (here, here.parent) if (d / "study.json").is_file() and (d / "instance.json").is_file()),
    Path(os.environ.get("LATBENCH_RESULTS", "/wrk/z/lab-kernel-latency/results/kernel-latency")) / "st-kconfig-factors@nucleo_g474re/st-stack-guard"))
lab = next((d for d in INSTANCE.resolve().parents if (d / ".lab" / "config").is_file()), None)
if os.environ.get("LATBENCH_REPO"):
    REPO = Path(os.environ["LATBENCH_REPO"])
elif lab:
    cfg = configparser.ConfigParser()
    cfg.read(lab / ".lab" / "config")
    REPO = lab / cfg.get("design", "repo", fallback="safety")
else:
    REPO = Path(subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True,
                               check=True).stdout.strip())

In [ ]:
import pandas as pd

sys.path.insert(0, str(REPO / "scripts" / "probatrix" / "benchmarks"))
import data  # noqa: E402
import nbfig  # noqa: E402

pd.set_option("display.max_colwidth", None)
inst = data.load_instance(INSTANCE)
nbfig.intro(inst)

## Central results

In [ ]:
inst.central()

## Hypotheses

A hypothesis is one claim about a mechanism. It is wrong if one of its
checks is wrong. The statements and the limits were written before the
runs; the column `basis` says where each number comes from.

In [ ]:
inst.hypotheses()

In [ ]:
inst.checks()

## Background

The defaults of the apps have `MPU_STACK_GUARD=y`. PendSV updates the
MPU region of the stack guard at each thread switch. Each wake and each
`t2t` sample has one thread switch. The IRQ entry path has no scheduler
and no FP context.

The variant `no-mpu-guard` sets `HW_STACK_PROTECTION=n` and
`MPU_STACK_GUARD=n`. `MPU_STACK_GUARD=n` alone has no effect, because
`HW_STACK_PROTECTION=y` selects `MPU_STACK_GUARD` (`../wp3-variants.md`).

H1 expects 100..250 cycles less for the `isr2thr` and `t2t` p50
(checks b and c), and no change of the `irq` p50 (+-2 cycles, check
a).

In [ ]:
LOAD_NAMES = {"none": "no load", "busy16": "16 busy threads", "t64": "64 load timers"}


def diffs(exp):
    """Each point minus the point of the variant baseline with the same load, in the same run (nbfig.diffs)."""
    return nbfig.diffs(inst, exp, "variant", "baseline", match=("load",), names=LOAD_NAMES)

## no-mpu-guard minus the baseline

Each bar is the value of `no-mpu-guard` minus the value of `baseline`
in the same run, for each metric of the three experiments. A bar to the
left of 0 means that the variant is faster. The three panels give the
p50, the p99 and the max. Each panel has its own x scale. The colour is
the experiment. A grey band is the range that a check of H1 expects.
The table gives the same differences in cycles.

In [ ]:
EXPS = ['irq-kconfig-1m', 'wake-kconfig-1m', 'thread-kconfig-1m']
parts, lims = [], []
for e in EXPS:
    if not nbfig.built(inst, e):
        continue
    d = diffs(e)
    lim = nbfig.check_limits(inst, e, d)
    lims.append(lim.assign(y=lim.metric + " (" + e + ")", panel="no-mpu-guard"))
    d = d[d.variant == "no-mpu-guard"]
    parts.append(d.assign(y=d.metric + " (" + e + ")", experiment=e, panel="no-mpu-guard"))
sg = pd.concat(parts)
nbfig.diff_fig(sg, "no-mpu-guard minus baseline", limits=pd.concat(lims), y="y", color="experiment",
               menu="panel").show()

In [ ]:
nbfig.diff_table(sg.assign(metric="no-mpu-guard - baseline"), y="y")

## The distributions

The CCDF gives the share of the samples of x or more, for `baseline`
and `no-mpu-guard`. The menu selects the metric.

In [ ]:
if nbfig.built(inst, "irq-kconfig-1m"):
    nbfig.ccdf(nbfig.hists_of(inst, "irq-kconfig-1m", points=["baseline", "no-mpu-guard"]), "irq-kconfig-1m: CCDF").show()

In [ ]:
if nbfig.built(inst, "wake-kconfig-1m"):
    nbfig.ccdf(nbfig.hists_of(inst, "wake-kconfig-1m", points=["baseline", "no-mpu-guard"]), "wake-kconfig-1m: CCDF").show()

In [ ]:
if nbfig.built(inst, "thread-kconfig-1m"):
    nbfig.ccdf(nbfig.hists_of(inst, "thread-kconfig-1m", points=["baseline", "no-mpu-guard"]), "thread-kconfig-1m: CCDF").show()